# Paper 12 补充实验 — Colab A100 运行脚本

三批实验：
- Step 1: Full Fine-Tuning baseline (1 exp, ~30 min)
- Step 2: LoRA split-QKV 消融 (6 exp, ~2.2h)
- Step 3: BigEarthNet 验证 (18 exp, ~4.8h)

建议分 2 个 session：Session 1 跑 Step 1+2，Session 2 跑 Step 3。

**前置条件**：
1. GitHub 仓库已 push（Colab 会 clone master 分支）
2. Google Drive 已挂载（用于保存结果和缓存 Prithvi 权重）
3. Prithvi_100M.pt 提前上传到 Drive（或首次运行自动下载）

## 0. 环境准备

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
RESULTS_DIR = '/content/drive/MyDrive/paper12_results'
WEIGHTS_DIR = '/content/drive/MyDrive/paper12_weights'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print(f'Results: {RESULTS_DIR}')
print(f'Weights: {WEIGHTS_DIR}')

## 1. 克隆仓库 + 安装依赖

In [ ]:
%cd /content
!rm -rf AlphaEarth-System
!git clone https://github.com/zhouning/alphaearth-training-system.git AlphaEarth-System
%cd /content/AlphaEarth-System
!git log --oneline -5

In [ ]:
!pip install -q -e '.[bench]'
!pip install -q torchgeo

## 2. 下载 / 复用 Prithvi-100M 权重

优先从 Drive 缓存读取，缓存没有则从 HuggingFace 下载后缓存到 Drive。

In [ ]:
import os, shutil

CACHED = f'{WEIGHTS_DIR}/Prithvi_100M.pt'
LOCAL  = '/content/AlphaEarth-System/data/weights/prithvi/Prithvi_100M.pt'
os.makedirs(os.path.dirname(LOCAL), exist_ok=True)

if os.path.exists(CACHED):
    print(f'Using cached weights from Drive: {CACHED}')
    shutil.copy(CACHED, LOCAL)
else:
    print('Downloading Prithvi-100M from HuggingFace...')
    !pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download(repo_id='ibm-nasa-geospatial/Prithvi-100M', filename='Prithvi_100M.pt')
    shutil.copy(downloaded, LOCAL)
    shutil.copy(downloaded, CACHED)
    print(f'Cached to Drive: {CACHED}')

print(f'Weights ready at: {LOCAL}')
print(f'Size: {os.path.getsize(LOCAL)/1e6:.1f} MB')

In [ ]:
# 注入 checkpoint 路径到所有 YAML 配置
import yaml, glob
for cfg_path in glob.glob('geoadapter/bench/configs/*.yaml'):
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    if 'prithvi' in cfg:
        cfg['prithvi']['checkpoint'] = LOCAL
        with open(cfg_path, 'w') as f:
            yaml.safe_dump(cfg, f, sort_keys=False)
        print(f'Patched: {cfg_path}')

In [ ]:
# 验证 backbone 能加载权重
import torch
from geoadapter.models.prithvi import PrithviBackbone
bb = PrithviBackbone(pretrained=True, checkpoint_path=LOCAL)
print(f'Backbone params: {sum(p.numel() for p in bb.parameters())/1e6:.1f}M')
x = torch.randn(2, 6, 64, 64)
print(f'Forward pass output: {bb(x).shape}')

## 3. Session 1 — Step 1: Full Fine-Tuning (~30 min)

In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
out = f'{RESULTS_DIR}/full_finetune_{ts}.json'
!python -m geoadapter.bench.run_benchmark \
    --config geoadapter/bench/configs/full_finetune.yaml \
    --output {out} 2>&1 | tee {RESULTS_DIR}/full_finetune_{ts}.log
print(f'\n=> Results: {out}')

## 4. Session 1 — Step 2: LoRA split-QKV 消融 (~2.2h)

In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
out = f'{RESULTS_DIR}/lora_ablation_{ts}.json'
!python -m geoadapter.bench.run_benchmark \
    --config geoadapter/bench/configs/lora_ablation.yaml \
    --output {out} 2>&1 | tee {RESULTS_DIR}/lora_ablation_{ts}.log
print(f'\n=> Results: {out}')

## 5. Session 2 — Step 3: BigEarthNet 验证 (~4.8h)

**注意**：BigEarthNet-S2 全量 ~66GB，首次运行 torchgeo 会自动下载。建议：
- Colab 本地 SSD ~100GB 够用，但重启 session 后会清空
- 可选：提前下载到 Drive 并挂载为 dataset_root
- 当前配置用 `max_samples: 50000` 限制训练规模，实际加载仍需完整数据集

In [ ]:
# 可选：把 dataset_root 指向 Drive（避免重新下载）
BEN_ROOT = f'{WEIGHTS_DIR}/../bigearthnet'
os.makedirs(BEN_ROOT, exist_ok=True)

import yaml
cfg_path = 'geoadapter/bench/configs/bigearthnet.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)
cfg['experiment']['dataset_root'] = BEN_ROOT
with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(f'dataset_root set to: {BEN_ROOT}')

In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
out = f'{RESULTS_DIR}/bigearthnet_{ts}.json'
!python -m geoadapter.bench.run_benchmark \
    --config geoadapter/bench/configs/bigearthnet.yaml \
    --output {out} 2>&1 | tee {RESULTS_DIR}/bigearthnet_{ts}.log
print(f'\n=> Results: {out}')

## 6. 结果汇总

In [ ]:
import json, glob
import pandas as pd

all_results = []
for jf in sorted(glob.glob(f'{RESULTS_DIR}/*.json')):
    with open(jf) as f:
        rows = json.load(f)
    for r in rows:
        r['batch'] = os.path.basename(jf).replace('.json','')
        all_results.append(r)

df = pd.DataFrame(all_results)
print(df.to_string())
df.to_csv(f'{RESULTS_DIR}/summary.csv', index=False)
print(f'\nSummary saved to: {RESULTS_DIR}/summary.csv')

In [ ]:
# 按 (method, modality) 聚合
metric_col = 'overall_accuracy' if 'overall_accuracy' in df.columns else 'mAP'
if metric_col in df.columns:
    agg = df.groupby(['method','modality'])[metric_col].agg(['mean','std','count']).round(4)
    print(agg)